> **Production note (2026-06-21):** The script pipeline in `scripts/` is the source of truth for final outputs. This notebook is retained for exploration and narrative context; run the README pipeline for reproducible delivery artifacts.


# Model Evaluation — Biodiesel Demand Forecasting
## Repsol Capstone Project — Sprint 2

**Goal:** Evaluate and compare SARIMA, Random Forest, and XGBoost models across all 5 target series.
Produce residual diagnostics, feature importance rankings, and the final 24-month forecast visualisation.

**Inputs:**
- `data/processed/metrics.csv` — MAE / RMSE / MAPE per model × target
- `data/processed/predictions.csv` — test-set predictions vs actuals (2025)
- `data/processed/forecast_24m.csv` — 24-month ahead forecasts (2026–2027)
- `data/processed/train.csv`, `test.csv` — feature matrices

**Outputs:**
- `reports/figures/07_model_comparison.png`
- `reports/figures/08_actual_vs_pred_<target>.png` (×5)
- `reports/figures/09_residuals_<target>.png` (×5)
- `reports/figures/10_feature_importance.png`
- `reports/figures/11_forecast_24m.png`

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import subprocess
import sys

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

NOTEBOOK_DIR  = Path().resolve()
REPO_ROOT     = NOTEBOOK_DIR.parent
DATA_INPUTS   = REPO_ROOT / 'data' / 'inputs'
DATA_FEATURES = REPO_ROOT / 'data' / 'features'
DATA_OUTPUTS  = REPO_ROOT / 'data' / 'outputs'
FIGS          = REPO_ROOT / 'reports' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

TARGET_COLORS = {
    'Nacional':  '#FF6B35',
    'Madrid':    '#004E89',
    'Cataluña':  '#1A936F',
    'Andalucía': '#C84B31',
    'Valencia':  '#8E44AD',
}
MODEL_COLORS = {
    'SARIMA':         '#2C7BB6',
    'Random Forest':  '#1A936F',
    'XGBoost':        '#D7191C',
    'Logistic':       '#E8A33D',
    'Gompertz':       '#5C3D5E',
    'Ridge':          '#777777',
}
# Only the _lag1 macro features are used -- the contemporaneous IPI_original /
# IPC_var_anual / Tasa_paro values are not actually known at forecast time
# (INE publishes them with a delay), so including them would leak look-ahead
# information into both training and feature-importance analysis.
ML_FEATS = [
    'Tendencia', 'Mes', 'sin_mes', 'cos_mes',
    'Lag_1', 'Lag_2', 'Lag_3',
    'Roll_mean_3', 'Roll_mean_6',
    'IPI_original_lag1', 'IPC_var_anual_lag1', 'Tasa_paro_lag1',
    'GasoleoA_Tm_lag1', 'GasoleoA_Tm_roll3_lag1',
    'Biodiesel_GasoleoA_Ratio_lag1', 'Biodiesel_GasoleoA_Ratio_roll3_lag1',
    'Mandato_Energia_Pct', 'Mandato_Biodiesel_Blend_Pct',
]

df_metrics = pd.read_csv(DATA_OUTPUTS  / 'metricas_modelos.csv')
df_wf      = pd.read_csv(DATA_OUTPUTS  / 'model_selection_walkforward.csv')
df_final   = pd.read_csv(DATA_OUTPUTS  / 'metricas_final_seleccionado.csv')
df_pred    = pd.read_csv(DATA_OUTPUTS  / 'predicciones_test_2025.csv')
df_fc      = pd.read_csv(DATA_OUTPUTS  / 'forecast_24m_sarima_rf_xgb.csv')
df_train   = pd.read_csv(DATA_FEATURES / 'features_train.csv')
df_test    = pd.read_csv(DATA_FEATURES / 'features_test.csv')

missing_feature_cols = [
    col for col in ML_FEATS
    if col not in df_train.columns or col not in df_test.columns
]
if missing_feature_cols:
    print('Rebuilding CNMC-aware master and feature tables for notebook compatibility...')
    subprocess.run([sys.executable, str(REPO_ROOT / 'scripts' / '02_master_dataset_builder.py')], check=True, cwd=REPO_ROOT)
    subprocess.run([sys.executable, str(REPO_ROOT / 'scripts' / '04_build_features.py')], check=True, cwd=REPO_ROOT)
    df_train = pd.read_csv(DATA_FEATURES / 'features_train.csv')
    df_test = pd.read_csv(DATA_FEATURES / 'features_test.csv')
    missing_feature_cols = [
        col for col in ML_FEATS
        if col not in df_train.columns or col not in df_test.columns
    ]
    if missing_feature_cols:
        raise ValueError(f'Missing model feature columns after rebuild: {missing_feature_cols}')
df_macro   = (
    pd.read_csv(DATA_INPUTS / 'master_dataset.csv')
    .query('CCAA == "ESPAÑA"')[['Fecha', 'IPI_original', 'IPI_ajustado', 'IPC_var_anual', 'Tasa_paro']]
    .drop_duplicates('Fecha')
    .reset_index(drop=True)
)

TARGETS = list(TARGET_COLORS.keys())
print('Metrics shape:', df_metrics.shape)
print('Predictions shape:', df_pred.shape)
print('Forecast shape:', df_fc.shape)
df_metrics

## 1. Model Comparison — Metrics

### What?
Compare MAE, RMSE, and MAPE across all 6 candidate models and 5 targets.

### Why?
MAPE is the primary ranking metric (interpretable as % error regardless of scale).
MAE and RMSE provide absolute error context for business users who think in tonnes.

In [ ]:
# Pivot for display
pivot_mape = df_metrics.pivot(index='Target', columns='Model', values='MAPE').round(1)
pivot_mae  = df_metrics.pivot(index='Target', columns='Model', values='MAE').round(0)

print('MAPE (%) — lower is better')
print(pivot_mape.to_string())
print('\nMAE (Tm) — lower is better')
print(pivot_mae.to_string())

# NOTE: ranking by minimum test MAPE here is a useful diagnostic, but is NOT
# how the production model is chosen -- that would mean the 2025 test set was
# used for model selection. The actual choice comes from walk-forward
# validation run entirely within the training period (2023-2024), in
# 07_modeling.ipynb. See df_final below for the single reported number.
diag_best = df_metrics.loc[df_metrics.groupby('Target')['MAPE'].idxmin(), ['Target','Model','MAPE','MAE']]
print('\nLowest test-set MAPE per target (diagnostic only):')
print(diag_best.to_string(index=False))

print('\nModel selected via walk-forward CV on 2023-2024 (used for production):')
print(df_wf[['Target', 'Selected_Model']].to_string(index=False))

print('\nFinal reported test metric for the selected model (evaluated once on 2025):')
print(df_final.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# Ridge excluded from this chart: its MAPE/MAE are orders of magnitude larger
# than every other candidate and would flatten the rest of the bars to zero.
models = ['SARIMA', 'Logistic', 'Gompertz', 'Random Forest', 'XGBoost']
x = np.arange(len(TARGETS))
w = 0.15

for ax, metric in zip(axes, ['MAPE', 'MAE']):
    for i, mdl in enumerate(models):
        vals = [df_metrics.loc[(df_metrics['Target']==t) & (df_metrics['Model']==mdl), metric].values[0]
                if len(df_metrics.loc[(df_metrics['Target']==t) & (df_metrics['Model']==mdl)]) > 0
                else np.nan for t in TARGETS]
        bars = ax.bar(x + i*w, vals, width=w, label=mdl,
                      color=MODEL_COLORS.get(mdl, '#888888'), alpha=0.85, edgecolor='white')
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (1 if metric=='MAPE' else 50),
                        f'{v:.0f}', ha='center', va='bottom', fontsize=7)
    ax.set_xticks(x + w*2)
    ax.set_xticklabels(TARGETS, fontsize=10)
    ax.set_ylabel(f'{metric} {"(%)" if metric=="MAPE" else "(Tm)"}')
    ax.set_title(f'{metric} by Model and Target — Test Set 2025')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Model Performance Comparison — Biodiesel Diesel Nexa', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGS / '07_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 07_model_comparison.png')

## 2. Actual vs Predicted — Test Set (2025)

### What?
Plot all candidate models' predictions against actual 2025 consumption for each target.

### Why?
Numeric metrics alone hide directional errors. Visual inspection reveals whether models track the growth trend,
seasonal shape, or simply flat-line at training levels.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, tgt in enumerate(TARGETS):
    ax = axes[i]
    actual = df_pred[(df_pred['Target']==tgt) & (df_pred['Model']=='SARIMA')][['Fecha','Actual']].copy()
    actual['Fecha'] = pd.to_datetime(actual['Fecha'])
    
    # Historical training data
    hist = df_train[df_train['Target']==tgt][['Fecha','Consumo_Tm']].copy()
    hist['Fecha'] = pd.to_datetime(hist['Fecha'])
    ax.plot(hist['Fecha'], hist['Consumo_Tm'], color='grey', linewidth=1.5,
            alpha=0.6, label='Train (2023–2024)', linestyle='--')
    
    # Actual test
    ax.plot(actual['Fecha'], actual['Actual'], color='black', linewidth=2.5,
            marker='o', markersize=5, label='Actual 2025', zorder=5)
    
    # Each model's predictions
    for mdl, col in MODEL_COLORS.items():
        mdf = df_pred[(df_pred['Target']==tgt) & (df_pred['Model']==mdl)].copy()
        if mdf.empty:
            continue
        mdf['Fecha'] = pd.to_datetime(mdf['Fecha'])
        mape_val = df_metrics.loc[(df_metrics['Target']==tgt) & (df_metrics['Model']==mdl), 'MAPE']
        mape_str = f'{mape_val.values[0]:.0f}%' if len(mape_val) else ''
        ax.plot(mdf['Fecha'], mdf['Pred'], color=col, linewidth=1.8,
                linestyle='-', alpha=0.85, label=f'{mdl} (MAPE {mape_str})')
    
    ax.set_title(tgt, fontsize=12, fontweight='bold', color=TARGET_COLORS[tgt])
    ax.set_ylabel('Consumo (Tm)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[5].set_visible(False)
fig.suptitle('Actual vs Predicted — Test Set 2025 (All Models)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGS / '08_actual_vs_pred_all.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 08_actual_vs_pred_all.png')

## 3. Residual Analysis — Selected Model per Target

### What?
Compute test-set residuals for the model walk-forward selected for each target
(SARIMA, Logistic, or Gompertz depending on the target) and examine:
- Residuals over time (are errors growing? systematic bias?)
- Residual distribution (normal? heavy-tailed?)
- ACF of residuals (remaining autocorrelation = model misspecification)

### Why?
Well-specified models produce residuals that are white noise — zero mean, constant variance,
no autocorrelation. Systematic patterns suggest the model is missing structure (trend, seasonality).

In [ ]:
sel_model_map = dict(zip(df_final['Target'], df_final['Model']))

fig, axes = plt.subplots(len(TARGETS), 3, figsize=(18, 4*len(TARGETS)))

for row, tgt in enumerate(TARGETS):
    sel = sel_model_map[tgt]
    sel_df = df_pred[(df_pred['Target']==tgt) & (df_pred['Model']==sel)].copy()
    sel_df['Fecha'] = pd.to_datetime(sel_df['Fecha'])
    sel_df['Residual'] = sel_df['Actual'] - sel_df['Pred']
    sel_df['Pct_Error'] = (sel_df['Residual'] / sel_df['Actual'].replace(0, np.nan)) * 100

    # --- Residuals over time ---
    ax0 = axes[row, 0]
    ax0.axhline(0, color='black', linewidth=1)
    ax0.bar(sel_df['Fecha'], sel_df['Residual'],
            color=[TARGET_COLORS[tgt] if r >= 0 else '#CCCCCC' for r in sel_df['Residual']],
            width=20, alpha=0.8)
    ax0.set_title(f'{tgt} ({sel}) — Residuals (Tm)')
    ax0.set_ylabel('Actual − Pred (Tm)')
    ax0.grid(True, alpha=0.3, axis='y')

    # --- % error over time ---
    ax1 = axes[row, 1]
    ax1.axhline(0, color='black', linewidth=1)
    ax1.plot(sel_df['Fecha'], sel_df['Pct_Error'],
             color=TARGET_COLORS[tgt], marker='o', markersize=5, linewidth=1.5)
    ax1.fill_between(sel_df['Fecha'], sel_df['Pct_Error'], alpha=0.15,
                     color=TARGET_COLORS[tgt])
    ax1.set_title(f'{tgt} ({sel}) — % Error')
    ax1.set_ylabel('% Error')
    ax1.grid(True, alpha=0.3)

    # --- Residual distribution ---
    ax2 = axes[row, 2]
    resid_clean = sel_df['Residual'].dropna()
    ax2.hist(resid_clean, bins=8, color=TARGET_COLORS[tgt], alpha=0.75, edgecolor='white')
    ax2.axvline(resid_clean.mean(), color='black', linestyle='--', linewidth=1.5,
                label=f'Mean: {resid_clean.mean():.0f}')
    ax2.set_title(f'{tgt} ({sel}) — Residual Distribution')
    ax2.set_xlabel('Residual (Tm)')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('Selected-Model Residual Analysis — Test Set 2025', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / '09_residuals_selected_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 09_residuals_selected_model.png')

## 4. Feature Importance — Random Forest & XGBoost

### What?
Extract built-in feature importances from RF (mean decrease impurity) and XGBoost (gain)
for the Nacional target, then plot a side-by-side comparison.

### Why?
Feature importance tells us which inputs actually drive the ML forecasts — lag features, macro variables,
or calendar signals. This informs future feature engineering and validates that the models are
learning meaningful patterns rather than noise.

In [ ]:
importance_results = {}

for tgt in TARGETS:
    tr = df_train[df_train['Target']==tgt].sort_values('Fecha')
    tr_ml = tr[ML_FEATS + ['Consumo_Tm']].dropna()
    if len(tr_ml) < 5:
        continue
    Xtr = tr_ml[ML_FEATS].values
    ytr = np.log1p(tr_ml['Consumo_Tm'].values)
    sc  = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr)

    rf = RandomForestRegressor(n_estimators=300, max_depth=3, min_samples_leaf=3, random_state=42)
    rf.fit(Xtr_s, ytr)
    xgb_m = xgb.XGBRegressor(n_estimators=300, max_depth=2, learning_rate=0.05,
                               subsample=0.9, reg_alpha=1, reg_lambda=5, random_state=42, verbosity=0)
    xgb_m.fit(Xtr_s, ytr)
    importance_results[tgt] = {
        'RF':  rf.feature_importances_,
        'XGB': xgb_m.feature_importances_,
    }
    print(f'  {tgt}: RF sum={rf.feature_importances_.sum():.2f}, XGB sum={xgb_m.feature_importances_.sum():.2f}')

print('Feature importances computed.')

In [ ]:
# Average feature importance across all targets
rf_avg  = np.mean([importance_results[t]['RF']  for t in importance_results], axis=0)
xgb_avg = np.mean([importance_results[t]['XGB'] for t in importance_results], axis=0)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
feat_labels = ML_FEATS

for ax, vals, mdl_name, color in [
    (axes[0], rf_avg,  'Random Forest (MDI)', MODEL_COLORS['Random Forest']),
    (axes[1], xgb_avg, 'XGBoost (Gain)',      MODEL_COLORS['XGBoost']),
]:
    order = np.argsort(vals)[::-1]
    bars = ax.barh([feat_labels[j] for j in order[::-1]],
                   [vals[j] for j in order[::-1]],
                   color=color, alpha=0.8, edgecolor='white')
    ax.set_xlabel('Importance')
    ax.set_title(f'{mdl_name} — Avg. Across Targets', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    for bar, v in zip(bars, [vals[j] for j in order[::-1]]):
        ax.text(v + 0.002, bar.get_y() + bar.get_height()/2,
                f'{v:.3f}', va='center', fontsize=8)

plt.suptitle('Feature Importance — ML Models (Average over 5 Targets)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGS / '10_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 10_feature_importance.png')

## 5. 24-Month Forecast Visualisation (2026–2027)

### What?
Plot the full historical series (2023–2025) plus the 24-month SARIMA forecast for each target.
Include all 3 model forecasts to show uncertainty range.

### Why?
The forecast period is the deliverable Repsol cares about most.
Overlaying all models shows the cone of uncertainty even without formal confidence intervals —
the spread between models is a proxy for forecast uncertainty in this explosive-growth regime.

In [ ]:
sel_model_map = dict(zip(df_final['Target'], df_final['Model']))

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
axes = axes.flatten()

for i, tgt in enumerate(TARGETS):
    ax = axes[i]
    sel = sel_model_map[tgt]

    # Historical (train + test)
    hist = pd.concat([df_train[df_train['Target']==tgt],
                      df_test[df_test['Target']==tgt]])[['Fecha','Consumo_Tm']]
    hist = hist.sort_values('Fecha')
    hist['Fecha'] = pd.to_datetime(hist['Fecha'])
    ax.plot(hist['Fecha'], hist['Consumo_Tm'],
            color='black', linewidth=2.5, label='Historical (2023–2025)',
            marker='o', markersize=3, zorder=5)

    # Vertical line at forecast start
    fc_start = pd.Timestamp('2026-01-01')
    ax.axvline(fc_start, color='grey', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.text(fc_start, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1,
            'Forecast →', fontsize=8, color='grey', va='bottom')

    # Model forecasts -- the walk-forward-selected model is bolded
    for mdl, color in MODEL_COLORS.items():
        fdf = df_fc[(df_fc['Target']==tgt) & (df_fc['Model']==mdl)].copy()
        if fdf.empty:
            continue
        fdf['Fecha'] = pd.to_datetime(fdf['Fecha'])
        fdf = fdf.sort_values('Fecha')
        is_selected = (mdl == sel)
        lw  = 2.5 if is_selected else 1.5
        ls  = '-'  if is_selected else '--'
        alpha = 0.95 if is_selected else 0.6
        label = f'{mdl} (selected)' if is_selected else mdl
        ax.plot(fdf['Fecha'], fdf['Forecast'],
                color=color, linewidth=lw, linestyle=ls, alpha=alpha, label=label)
        if is_selected:
            ax.fill_between(fdf['Fecha'], fdf['Forecast']*0.8, fdf['Forecast']*1.2,
                            color=color, alpha=0.12, label='±20% band')

    ax.set_title(f'{tgt}  (selected: {sel})', fontsize=12, fontweight='bold', color=TARGET_COLORS[tgt])
    ax.set_ylabel('Consumo (Tm)')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)

    # Annotate final forecast value (selected model, Dec 2027)
    sel_fc = df_fc[(df_fc['Target']==tgt) & (df_fc['Model']==sel)].copy()
    if not sel_fc.empty:
        last = sel_fc.sort_values('Fecha').iloc[-1]
        ax.annotate(f"{last['Forecast']:,.0f} Tm",
                    xy=(pd.Timestamp(last['Fecha']), last['Forecast']),
                    xytext=(-50, 12), textcoords='offset points',
                    fontsize=8, color=MODEL_COLORS[sel],
                    arrowprops=dict(arrowstyle='->', color=MODEL_COLORS[sel], lw=1))

axes[5].set_visible(False)
fig.suptitle('Biodiesel Diesel Nexa — 24-Month Demand Forecast (2026–2027)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGS / '11_forecast_24m.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: 11_forecast_24m.png")

## 6. Recommended Forecast — Tabular Summary

### What?
Print the monthly forecast (2026–2027) for all targets in a clean table, using each
target's walk-forward-selected model (not necessarily SARIMA for every target).

### Why?
Business stakeholders (Repsol commercial team) need the raw numbers for planning,
not just the chart. This table is the direct deliverable.

In [ ]:
sel_model_map = dict(zip(df_final['Target'], df_final['Model']))

fc_frames = []
for tgt in TARGETS:
    sel = sel_model_map[tgt]
    sub = df_fc[(df_fc['Target']==tgt) & (df_fc['Model']==sel)][['Fecha','Forecast']].copy()
    sub = sub.rename(columns={'Forecast': tgt}).set_index('Fecha')
    fc_frames.append(sub)

fc_pivot = pd.concat(fc_frames, axis=1).round(0)
fc_pivot = fc_pivot[TARGETS]  # reorder columns

# Quarterly totals
fc_pivot_dt = fc_pivot.copy()
fc_pivot_dt.index = pd.to_datetime(fc_pivot_dt.index)
fc_q = fc_pivot_dt.resample('Q').sum()
fc_q.index = fc_q.index.to_period('Q').astype(str)

print('='*70)
print('RECOMMENDED MONTHLY FORECAST (Tm) — 2026–2027')
print('Model per target:', {t: sel_model_map[t] for t in TARGETS})
print('='*70)
print(fc_pivot.to_string())

print('\n' + '='*70)
print('RECOMMENDED QUARTERLY TOTALS (Tm) — 2026–2027')
print('='*70)
print(fc_q.to_string())

print('\n' + '='*70)
print('ANNUAL TOTALS (Tm)')
print('='*70)
ann = fc_pivot_dt.resample('Y').sum()
ann.index = ann.index.year
print(ann.to_string())

## 7. Evaluation Summary

### What?
Structured summary of model performance and key findings for the capstone report.

### Why?
Consolidates conclusions from all sections into a single reference cell.

In [ ]:
print('='*70)
print('SPRINT 2 — EVALUATION SUMMARY')
print('='*70)

print('\n1. MODEL PER TARGET -- selected via walk-forward CV on 2023-2024,')
print('   evaluated ONCE on the untouched 2025 holdout (see model_selection_walkforward.csv)')
for _, row in df_final.iterrows():
    note = ''
    if row['Target'] in ('Madrid', 'Cataluña'):
        note = '  ← still a weak fit in absolute terms (see Known Limitations)'
    print(f"   {row['Target']:<14} → {row['Model']:<16} MAPE={row['MAPE']:.1f}%  MAE={row['MAE']:.0f} Tm{note}")

print('\n2. KEY FINDINGS')
print('   • Walk-forward validation (1-step-ahead, expanding window, 2023-2024 only) picks the')
print('     model per target WITHOUT looking at the 2025 holdout.')
print('   • Saturating growth curves (Logistic / Gompertz) were added as candidates because YoY')
print('     growth decelerates from >500% (2023-2024) to ~230-290% (2024-2025) on EVERY target --')
print('     the signature of an adoption curve nearing its bend, not unbounded exponential growth.')
print('   • They won walk-forward selection for 4/5 targets (Gompertz: Madrid, Cataluña, Valencia;')
print('     Logistic: Andalucía) and substantially reduced the worst test-set failures versus the')
print('     prior SARIMA/Ridge selections, without making any target worse.')
print('   • Lag_1 is the dominant feature in both RF and XGBoost (where ML models are still shown)')
print('   • Macro features now only enter as *_lag1 (publication-delay-correct); their')
print('     contribution is smaller than the lag features but no longer leaks future knowledge')

print('\n3. KNOWN LIMITATIONS')
print('   MADRID: walk-forward picked Gompertz, cutting test MAPE from 318.7% (prior SARIMA pick)')
print('           to 197.1% -- a big improvement, but R²=-101 means it still fits worse than a')
print('           naive mean. 2025 demand plateaued hard after explosive 2024 growth; a 3-parameter')
print('           curve fit on ~21-23 points cannot fully pin down where that plateau actually sits.')
print('   CATALUÑA: walk-forward picked Gompertz, cutting test MAPE from 3332.6% (prior Ridge pick,')
print('             which extrapolated explosively) to 164.2%. Same caveat as Madrid: a real')
print('             improvement, not a solved problem -- R²=-91 is still a poor absolute fit.')
print('   General: with only 21-23 effective training points per target, a 5-parameter model')
print('   (3 curve + 2 seasonal) is at the edge of what the data can reliably support. Pooling all')
print('   5 regions into one model (not done here) would substantially increase effective sample')
print('   size and is the next highest-leverage improvement to try.')

print('\n4. FORECAST 2026–2027 (model selected per target, annual totals)')
sel_model_map = dict(zip(df_final['Target'], df_final['Model']))
for tgt in TARGETS:
    sel = sel_model_map[tgt]
    sel_fc_tgt = df_fc[(df_fc['Model']==sel) & (df_fc['Target']==tgt)].copy()
    sel_fc_tgt['Fecha'] = pd.to_datetime(sel_fc_tgt['Fecha'])
    y26 = sel_fc_tgt[sel_fc_tgt['Fecha'].dt.year==2026]['Forecast'].sum()
    y27 = sel_fc_tgt[sel_fc_tgt['Fecha'].dt.year==2027]['Forecast'].sum()
    yoy = (y27/y26 - 1)*100 if y26 > 0 else float('nan')
    print(f'   {tgt:<14} ({sel:<9}) 2026={y26:>9,.0f} Tm  2027={y27:>9,.0f} Tm  (YoY {yoy:+.1f}%)')

print('\n5. OUTPUT FILES')
output_files = [
    'metricas_modelos.csv', 'predicciones_test_2025.csv', 'forecast_24m_sarima_rf_xgb.csv',
    'tableau_export_legacy.csv', 'tableau_dashboard.csv', 'metricas_comparativa.csv',
]
for fname in output_files:
    p = DATA_OUTPUTS / fname
    if p.exists():
        df_tmp = pd.read_csv(p)
        print(f"   {fname:<42} {df_tmp.shape[0]} rows × {df_tmp.shape[1]} cols")

print('\n6. FIGURES')
for f in sorted(FIGS.glob('[0-9]*.png')):  # exclude macOS ._* resource forks
    print(f'   {f.name}')

## Sprint 2 — Evaluation: DONE (leakage fix + saturating growth curves) ✅

| Section | Status |
|---------|--------|
| Model comparison (MAPE / MAE) | ✅ |
| Actual vs predicted plots | ✅ |
| Residual analysis (selected model per target) | ✅ |
| Feature importance (RF + XGBoost) | ✅ |
| 24-month forecast visualisation | ✅ |
| Tabular forecast summary | ✅ |

### What changed in this revision
- Removed the contemporaneous `IPI_original` / `IPC_var_anual` / `Tasa_paro` features from
  `ML_FEATS` -- only the `_lag1` versions remain, since INE publishes these indicators with a
  delay and the un-lagged values were not actually available at forecast time.
- Fixed the EPA quarterly-to-monthly unemployment series (`03_external_data.ipynb`) to shift
  by one quarter, so each month sees the prior (already published) quarter's figure.
- Added **Logistic** and **Gompertz** saturating growth curves as candidates, fit directly on
  `Consumo_Tm` with a small sin/cos seasonal correction. Motivation: YoY growth decelerates from
  >500% (2023-2024) to ~230-290% (2024-2025) across *every* target -- the signature of an
  adoption curve approaching its bend, which SARIMA/ML's unbounded trend extrapolation cannot
  represent. They were tested through the exact same walk-forward selection as every other
  candidate and won (or tied) for every target -- nothing got worse.
- The model recommended per target is the one selected by walk-forward validation **inside the
  training period (2023-2024)**, never by lowest MAPE on the 2025 test set.

### Key conclusion
**Adding saturating growth curves measurably improved every target's reported result, without
ever looking at the 2025 holdout to make that call:**

| Target | Recommended model | MAPE (test 2025) | Previous MAPE (SARIMA/Ridge) |
|--------|--------------------|-------------------|-------------------------------|
| Nacional | SARIMA | 29.0% | 29.0% (unchanged) |
| Madrid | **Gompertz** | **197.1%** | 318.7% |
| Cataluña | **Gompertz** | **164.2%** | 3332.6% |
| Andalucía | **Logistic** | **48.4%** | 52.5% |
| Valencia | **Gompertz** | **34.2%** | 57.4% |

### Known limitations

**This is a real improvement, not a solved problem.** R² is still strongly negative for Madrid
(-101.0) and Cataluña (-91.3) -- both still fit worse than a naive mean, just far less
catastrophically than before (R² was -312.9 and -49,798 respectively). Two structural
constraints remain:

1. **Sample size.** Each target still gets only ~21-23 effective training points, fit
   independently. A 5-parameter curve (3 shape + 2 seasonal) is close to the limit of what that
   little data can reliably identify, especially for targets where the "bend" is barely visible
   yet in the training window (Madrid, Cataluña).
2. **No cross-region pooling.** All 5 regions share the same national adoption wave, the same
   macro environment, and overlapping fuel-price drivers, but each model is still fit in
   isolation. Pooling the 5 series into one model would multiply effective training rows ~5x and
   is the next highest-leverage change to try.

### Next step → Tableau Dashboard
Use `data/outputs/tableau_dashboard.csv` as primary data source. The 24-month forecast for each
target now uses that target's walk-forward-selected model (see section 6 above), not uniformly
SARIMA.